In [1]:
import pandas as pd

# Đọc dữ liệu từ file parquet
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

# Chuyển đổi timestamp sang số (giây)
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

# Sắp xếp tĩnh trực tiếp theo thời gian
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

print("Dữ liệu đã được load và sắp xếp theo thời gian thành công!")

Dữ liệu đã được load và sắp xếp theo thời gian thành công!


In [2]:
def check_time_window_numeric(df, delta_t_seconds):
    """
    delta_t_seconds: Khoảng thời gian muốn thử (tính bằng giây).
    Ví dụ: 0.01 là 10 mili-giây, 0.1 là 100 mili-giây.
    """
    # Gom nhóm bằng cách chia lấy phần nguyên
    # Ép kiểu int để loại bỏ phần dư, biến nó thành 1 ID duy nhất cho mỗi cửa sổ
    window_ids = (df['timestamp_num'] / delta_t_seconds).astype(int)
    
    # Đếm số lượng flow trong mỗi ID (chính là 1 snapshot)
    flow_counts = window_ids.value_counts()
    
    print(f"--- Đánh giá cửa sổ Delta t = {delta_t_seconds} giây ---")
    print(f"Trung bình số flow / snapshot: {flow_counts.mean():.0f}")
    print(f"Lớn nhất (Đỉnh tấn công): {flow_counts.max()} flows")
    print(f"Nhỏ nhất (Lúc rảnh): {flow_counts.min()} flows")
    print(f"Tổng số snapshot sẽ tạo ra: {len(flow_counts)}")
    print("-" * 40)

# Chạy thử với 3 mốc mili-giây phổ biến:
check_time_window_numeric(train_df, 0.01) # Cửa sổ 10 ms (0.01s)
check_time_window_numeric(train_df, 0.05) # Cửa sổ 50 ms (0.05s)
check_time_window_numeric(train_df, 0.10) # Cửa sổ 100 ms (0.10s)

--- Đánh giá cửa sổ Delta t = 0.01 giây ---
Trung bình số flow / snapshot: 6
Lớn nhất (Đỉnh tấn công): 2670 flows
Nhỏ nhất (Lúc rảnh): 1 flows
Tổng số snapshot sẽ tạo ra: 403914
----------------------------------------
--- Đánh giá cửa sổ Delta t = 0.05 giây ---
Trung bình số flow / snapshot: 26
Lớn nhất (Đỉnh tấn công): 5835 flows
Nhỏ nhất (Lúc rảnh): 1 flows
Tổng số snapshot sẽ tạo ra: 96791
----------------------------------------
--- Đánh giá cửa sổ Delta t = 0.1 giây ---
Trung bình số flow / snapshot: 50
Lớn nhất (Đỉnh tấn công): 6796 flows
Nhỏ nhất (Lúc rảnh): 1 flows
Tổng số snapshot sẽ tạo ra: 49877
----------------------------------------


In [3]:
import pandas as pd
import numpy as np

def create_graph_snapshots_time_based(df, feature_cols, label_col='label', delta_t=0.1):
    """
    Tạo các snapshot đồ thị dựa trên cửa sổ thời gian tĩnh (Delta t).
    
    delta_t: Kích thước cửa sổ (tính bằng giây). Mặc định 0.1 = 100 mili-giây.
    """
    print(f"Đang băm dữ liệu theo cửa sổ thời gian Delta t = {delta_t} giây...")
    
    
    # 2. Tạo ID cho từng cửa sổ (Chia lấy phần nguyên)
    df['window_id'] = (df['timestamp_num'] / delta_t).astype(int)
    
    snapshots = []
    
    # 3. Gom nhóm (Groupby) các flow có chung Window ID lại thành 1 snapshot
    grouped = df.groupby('window_id')
    
    for window_id, group in grouped:
        x_snapshot = group[feature_cols].values
        y_snapshot = group[label_col].values
        
        meta_snapshot = {
            'dst_ip': group['dst_ip'].values,
            'dst_port': group['dst_port'].values,
            'timestamp': group['timestamp_num'].values,
            'global_indices': group.index.values 
        }
        
        snapshots.append({
            'x': x_snapshot,
            'y': y_snapshot,
            'meta': meta_snapshot
        })
        
    print(f"Hoàn tất! Đã tạo thành công {len(snapshots)} snapshots.")
    return snapshots

# ==========================================
# THỰC THI CHO TẬP TRAIN, VALID VÀ TEST
# ==========================================
# (Chạy lần lượt cho cả 3 tập để đồng bộ dữ liệu)
feature_cols = [col for col in train_df.columns if col not in ["flow_id",'timestamp', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'label', 'timestamp_num', "src_port", "dst_port"]]
train_snapshots_time = create_graph_snapshots_time_based(train_df, feature_cols, label_col='label', delta_t=0.1)
valid_snapshots_time = create_graph_snapshots_time_based(valid_df, feature_cols, label_col='label', delta_t=0.1)
test_snapshots_time = create_graph_snapshots_time_based(test_df, feature_cols, label_col='label', delta_t=0.1)

Đang băm dữ liệu theo cửa sổ thời gian Delta t = 0.1 giây...
Hoàn tất! Đã tạo thành công 49877 snapshots.
Đang băm dữ liệu theo cửa sổ thời gian Delta t = 0.1 giây...
Hoàn tất! Đã tạo thành công 9369 snapshots.
Đang băm dữ liệu theo cửa sổ thời gian Delta t = 0.1 giây...
Hoàn tất! Đã tạo thành công 13313 snapshots.


In [4]:
import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data
from tqdm import tqdm
def build_graph_from_snapshot(snapshot, k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Xử lý một snapshot đơn lẻ (Đã tối ưu CPU và chuẩn hóa cho Time-based).
    """
    # 1. ÉP KIỂU TENSOR NGAY TỪ ĐẦU (Rất quan trọng cho PyTorch Geometric)
    x = torch.tensor(snapshot['x'], dtype=torch.float32)
    y = torch.tensor(snapshot['y'], dtype=torch.long)
    meta = snapshot['meta']
    num_nodes = x.shape[0]

    # 2. CẮT SỚM (EARLY EXIT): Nếu chỉ có 1 node, trả về luôn đồ thị không cạnh
    # Giúp tiết kiệm hàng triệu chu kỳ CPU cho các snapshot lúc mạng rảnh rỗi
    if num_nodes <= 1:
        return Data(x=x, edge_index=torch.empty((2, 0), dtype=torch.long), 
                    edge_attr=torch.empty((0,), dtype=torch.float32), y=y)

    target_ids = np.array([f"{ip}:{port}" for ip, port in zip(meta['dst_ip'], meta['dst_port'])])
    timestamps = meta['timestamp']
    edge_index_list = []

    # 3. Vòng lặp nối cạnh Temporal K-NN (Logic của bạn rất chuẩn, giữ nguyên)
    for i in range(num_nodes):
        same_target_mask = (target_ids == target_ids[i])
        same_target_indices = np.where(same_target_mask)[0]
        same_target_indices = same_target_indices[same_target_indices != i]

        if len(same_target_indices) == 0:
            continue

        time_diffs = np.abs(timestamps[same_target_indices] - timestamps[i])
        # k_actual sẽ tự động bằng độ dài mảng nếu số node ít hơn k
        k_actual = min(k, len(time_diffs))
        top_k_local_indices = np.argsort(time_diffs)[:k_actual]
        top_k_global_indices = same_target_indices[top_k_local_indices]

        for j in top_k_global_indices:
            edge_index_list.append([i, j])

    # 4. Rào lỗi số 2: Có nhiều node nhưng không có node nào chung IP/Port
    if not edge_index_list:
        return Data(x=x, edge_index=torch.empty((2, 0), dtype=torch.long), 
                    edge_attr=torch.empty((0,), dtype=torch.float32), y=y)

    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()

    # 5. Tính Cosine Similarity (x lúc này đã là Torch Tensor nên tính toán an toàn)
    src_nodes_features = x[edge_index[0]]
    dst_nodes_features = x[edge_index[1]]
    cos_sim = F.cosine_similarity(src_nodes_features, dst_nodes_features, dim=1, eps=1e-8)
    cos_sim = (cos_sim + 1.0) / 2.0 

    # 6. Tính Time Decay với chuẩn hóa Min-Max cục bộ
    src_times = torch.tensor(timestamps[edge_index[0].numpy()], dtype=torch.float32)
    dst_times = torch.tensor(timestamps[edge_index[1].numpy()], dtype=torch.float32)
    time_diffs_tensor = torch.abs(src_times - dst_times)
    
    max_diff = time_diffs_tensor.max()
    if max_diff > 0:
        time_diffs_norm = time_diffs_tensor / max_diff
    else:
        time_diffs_norm = time_diffs_tensor
        
    time_decay = torch.exp(-lambda_decay * time_diffs_norm)
    edge_weight = alpha * cos_sim + beta * time_decay

    return Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y)


def convert_all_snapshots_to_graphs(snapshots_list, dataset_name="Train Set", k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Hàm wrapper bọc tqdm bên ngoài để track tiến trình tạo đồ thị cho TOÀN BỘ danh sách snapshots.
    """
    graph_objects = []
    
    # tqdm sẽ tự động tính toán % tiến độ, số snapshot/s và thời gian còn lại
    progress_bar = tqdm(
        snapshots_list, 
        desc=f"Building Graphs ({dataset_name})", 
        unit="snapshot",
        ncols=100 # Độ rộng của thanh tiến trình hiển thị trên console
    )
    
    for snap in progress_bar:
        graph_data = build_graph_from_snapshot(
            snapshot=snap, 
            k=k, 
            alpha=alpha, 
            beta=beta, 
            lambda_decay=lambda_decay
        )
        graph_objects.append(graph_data)
        
    return graph_objects

In [5]:
# Thực hiện tạo đồ thị và track tiến trình cho tập Train
train_graphs = convert_all_snapshots_to_graphs(train_snapshots_time, dataset_name="Train Set")

# Thực hiện tạo đồ thị và track tiến trình cho tập Valid
valid_graphs = convert_all_snapshots_to_graphs(valid_snapshots_time, dataset_name="Valid Set")

# Thực hiện tạo đồ thị và track tiến trình cho tập Test
test_graphs = convert_all_snapshots_to_graphs(test_snapshots_time, dataset_name="Test Set")

Building Graphs (Test Set): 100%|██████████████████████| 13313/13313 [00:18<00:00, 708.66snapshot/s]


In [6]:
from torch_geometric.loader import DataLoader

# Thiết lập Batch Size (Số lượng snapshot đưa vào GPU trong 1 bước)
# Với máy cấu hình thông thường, bạn có thể để 32 hoặc 64. 
BATCH_SIZE = 32 

# Khởi tạo DataLoader cho cả 3 tập
# Lưu ý: Tập Train cần shuffle=True để mô hình học khách quan, Valid/Test giữ nguyên thứ tự
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"Đã chuẩn bị xong DataLoader:")
print(f" - Train Batches: {len(train_loader)}")
print(f" - Valid Batches: {len(valid_loader)}")
print(f" - Test Batches: {len(test_loader)}")

Đã chuẩn bị xong DataLoader:
 - Train Batches: 1559
 - Valid Batches: 293
 - Test Batches: 417


In [7]:
import torch
import numpy as np

class EarlyStopping:
    """
    Trình quản lý dừng sớm (Early Stopping) để theo dõi Macro F1 trên tập Validation.
    """
    def __init__(self, patience=8, path='best_baseline_gcn.pt'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_f1 = -float('inf')
        self.early_stop = False

    def __call__(self, val_f1, model):
        # Nếu F1 cải thiện (lớn hơn điểm tốt nhất trước đó)
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.save_checkpoint(model)
            self.counter = 0 # Reset bộ đếm về 0
        else:
            # Nếu không cải thiện, tăng bộ đếm thêm 1
            self.counter += 1
            print(f' -> [EarlyStopping] Bộ đếm tăng: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        '''Lưu mô hình khi điểm Validation F1 tăng'''
        torch.save(model.state_dict(), self.path)
        print(f' -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.')

In [8]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score

def train_epoch(epoch, EPOCHS):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Train
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Train]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
        # Cập nhật số loss liên tục trên thanh tiến trình
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    loss_avg = total_loss / len(train_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

@torch.no_grad()
def valid_epoch(epoch, EPOCHS):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Valid
    progress_bar = tqdm(valid_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Valid]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
    loss_avg = total_loss / len(valid_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

In [9]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GATConv

class AdvancedGAT(torch.nn.Module):
    def __init__(self, num_node_features, num_classes=11):
        super(AdvancedGAT, self).__init__()
        
        # 1. Lớp đầu vào (Giữ nguyên để nén feature)
        self.lin_in = Linear(num_node_features, 32)
        
        # 2. Các lớp GAT với Multi-head Attention
        # Lớp 1: Input 32 -> 4 heads x 16 = 64 chiều (Tương đương GCNConv 64)
        self.gat1 = GATConv(
            in_channels=32, out_channels=16, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 2: Input 64 -> 4 heads x 32 = 128 chiều (Tương đương GCNConv 128)
        self.gat2 = GATConv(
            in_channels=64, out_channels=32, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 3: Input 128 -> 4 heads x 64. 
        # CHÚ Ý: Ở lớp GNN cuối cùng, ta dùng concat=False để tính trung bình các heads, 
        # gom lại thành 64 chiều chuẩn bị cho lớp Linear.
        self.gat3 = GATConv(
            in_channels=128, out_channels=64, heads=4, concat=False, edge_dim=1
        )
        
        # 3. Lớp phân loại đầu ra
        self.lin_out = Linear(64, num_classes)

    def forward(self, x, edge_index, edge_attr):
        # TIỂU XẢO KỸ THUẬT QUAN TRỌNG:
        # GATConv yêu cầu edge_attr phải là ma trận 2 chiều [Số cạnh, Số feature cạnh].
        # Trọng số Cosine + Time Decay của ta đang là mảng 1 chiều [E], ta cần thêm 1 trục (unsqueeze).
        if edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(-1)
            
        x = self.lin_in(x)
        x = F.relu(x)
        
        # Lớp GAT 1 (Dùng hàm ELU thay vì ReLU vì các paper về GAT chứng minh ELU mượt mà hơn cho Attention)
        x = self.gat1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x) 
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 2
        x = self.gat2(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 3
        x = self.gat3(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp Output
        x = self.lin_out(x)
        
        return F.log_softmax(x, dim=1)

In [11]:
# Khởi tạo mô hình

# Chuyển mô hình lên GPU (nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Chạy đoạn code sinh class_weights của bạn
# LƯU Ý: all_train_labels phải là nhãn của TẬP TRAIN (Không gộp Valid/Test vào để tránh Data Leakage)
import sklearn.utils.class_weight as class_weight

unique_classes = np.unique(train_df['label']) # Lấy các class có trong tập train
all_train_labels = train_df['label'].values

class_weights = class_weight.compute_class_weight(
    class_weight='balanced', 
    classes=unique_classes, 
    y=all_train_labels
)
class_weights = np.nan_to_num(class_weights, nan=1.0, posinf=10.0, neginf=1.0)
class_weights = np.power(class_weights, 0.4) 
class_weights = np.clip(class_weights, a_min=0.5, a_max=6.0) 

# Đưa lên thiết bị (GPU/CPU)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device) 

# ==========================================
# 2. Áp dụng vào Hàm Mất Mát (Loss Function)
# ==========================================
# Chèn class_weights_tensor vào tham số 'weight' của NLLLoss
criterion = torch.nn.NLLLoss(weight=class_weights_tensor)

print("Đã tích hợp Class Weights vào hàm Loss thành công!")
print(f"Trọng số các class: {class_weights}")


Đã tích hợp Class Weights vào hàm Loss thành công!
Trọng số các class: [1.64730877 0.5        3.7653502  1.29446194 1.67705869 3.80442181
 2.0250111  0.83822433 1.76304403 1.22605595 1.69183888]


In [15]:
# 1. Khởi tạo GAT Model

model = AdvancedGAT(num_node_features=314, num_classes=11)
model = model.to(device)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)
# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

print(model)
EPOCHS = 50

# Khởi tạo bộ quản lý Early Stopping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_baseline_gat_time.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    scheduler.step(valid_f1)  # Cập nhật learning rate dựa trên Macro F1 của tập Validation
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
    
model.load_state_dict(torch.load('best_baseline_gat_time.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)
Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.3556 - Macro F1: 0.7079 | Valid Loss: 0.1927 - Macro F1: 0.8834
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.1226 - Macro F1: 0.8859 | Valid Loss: 0.3001 - Macro F1: 0.8151
 -> [EarlyStopping] Bộ đếm tăng: 1 / 8


Epoch: 003/50 | Train Loss: 0.1003 - Macro F1: 0.9087 | Valid Loss: 0.1490 - Macro F1: 0.9071
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 021/50 | Train Loss: 0.0390 - Macro F1: 0.9637 | Valid Loss: 0.1504 - Macro F1: 0.8938
 -> [EarlyStopping] Bộ đếm tăng: 5 / 8


Epoch: 022/50 | Train Loss: 0.0424 - Macro F1: 0.9604 | Valid Loss: 0.1860 - Macro F1: 0.8913
 -> [EarlyStopping] Bộ đếm tăng: 6 / 8


Epoch: 023/50 | Train Loss: 0.0491 - Macro F1: 0.9575 | Valid Loss: 0.1825 - Macro F1: 0.9251
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


Epoch: 024/50 | Train Loss: 0.0403 - Macro F1: 0.9656 | Valid Loss: 0.1514 - Macro F1: 0.8975
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 24!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!


C:\Users\Admin\AppData\Local\Temp\ipykernel_18004\3902398716.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gat_time.pt

In [16]:
from sklearn.metrics import classification_report
@torch.no_grad()
def test_benchmark_time_based(loader, model):
    model.eval()
    
    all_preds = []
    all_labels = []
    all_original_indices = [] # Lưu lại index gốc của file CSV
    
    print("Đang chạy Benchmark trên tập Test (Thời gian thực - Delta t = 0.1s)...")
    for batch in tqdm(loader, desc="Testing", ncols=100):
        batch = batch.to(device)
        
        # Forward pass
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
        # Đọc index gốc từ file snapshot truyền qua
        # Do batch gom nhiều đồ thị, batch.ptr sẽ quản lý index nhưng meta thì ta lấy trực tiếp từ batch nếu có
        # Hoặc nếu bạn chạy batch_size=32, hãy đảm bảo cấu trúc data lưu đúng mảng numpy gốc
        if hasattr(batch, 'global_indices'):
            all_original_indices.extend(batch.global_indices)
            
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"\n[DONE] Tổng số flow thực tế được đánh giá: {len(all_labels)}")
    print(f"🏆 CHÍNH THỨC - TIME-BASED TEST MACRO F1: {macro_f1:.4f}")
    print("==================================================")
    print(classification_report(all_labels, all_preds, digits=4))
    
    return all_preds, all_labels

In [17]:
model.load_state_dict(torch.load('best_baseline_gat_time.pt'))
model = model.to(device)
test_preds, test_labels = test_benchmark_time_based(test_loader, model)

C:\Users\Admin\AppData\Local\Temp\ipykernel_18004\3762751709.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gat_time.pt'

Đang chạy Benchmark trên tập Test (Thời gian thực - Delta t = 0.1s)...


Testing: 100%|███████████████████████████████████████████████████| 417/417 [00:03<00:00, 136.14it/s]



[DONE] Tổng số flow thực tế được đánh giá: 760240
🏆 CHÍNH THỨC - TIME-BASED TEST MACRO F1: 0.8459
              precision    recall  f1-score   support

           0     0.7250    0.8602    0.7868     19846
           1     0.9986    1.0000    0.9993    484077
           2     0.3641    0.9531    0.5269      2515
           3     0.9938    0.8826    0.9349     36253
           4     0.7773    0.7185    0.7467     18979
           5     0.9919    0.9976    0.9947      2451
           6     0.7066    0.7436    0.7247     11847
           7     1.0000    0.9927    0.9963    107436
           8     0.6741    0.9950    0.8037     16746
           9     1.0000    0.7072    0.8285     41523
          10     0.9282    0.9984    0.9620     18567

    accuracy                         0.9624    760240
   macro avg     0.8327    0.8954    0.8459    760240
weighted avg     0.9705    0.9624    0.9636    760240

